# Restarting from HDF5 files

VeloxChem supports robust **restart** capabilities via binary **HDF5 (`.h5`)** files that capture the essential electronic‑structure state needed to resume or reuse calculations reproducibly and efficiently.

## What is stored in the restart file?

The restart file is designed to be **portable**, **compact**, and **self‑describing**. It typically contains:

- **System definition**
  - Cartesian geometry (atomic numbers, positions)
  - Charge and spin multiplicity

- **Basis and integral metadata**
  - Basis‑set and ECP identifiers per atom
  - Integral conventions
  - XC grid settings

- **SCF/DFT electronic state**
  - Total energies
  - Molecular‑orbital (MO) coefficients and orbital energies
  - Occupations (including fractional, if used)
  - Density matrix (α/β for unrestricted cases)
  - SCF convergence thresholds and last achieved residuals
  - Functional specification

- **Additional model context (when active)**
  - Implicit solvation model and parameters (e.g., CPCM/SMD)
  - Explicit solvation model and parameters (polarizable embedding)
  - External fields (if present) and unit specifications

- **Provenance and versioning**
  - VeloxChem version, build info, and I/O schema version
  - Host and timestamp metadata to aid reproducibility

## What can be restarted?

- **SCF/DFT single‑point energy**  
  Resume from a converged density and MO set for instant re‑evaluation (e.g., with tighter thresholds or different print options).

- **Geometry optimization & PES scans**  
  Reuse the last converged density/MOs at a new geometry as a high‑quality initial guess to accelerate SCF convergence along a path or during restarts after wall‑time limits.

- **Vibrational analysis (IR/Raman)**  
  Start the property step from a converged electronic state at the optimized geometry.

- **Spectroscopies (UV/vis, optical activity)**  
  Provide a clean ground‑state reference (MOs, occupations, density) for TDDFT and CPP runs to reduce set‑up time and ensure consistency of state definitions.

## Writing restart files

VeloxChem can write a restart file at the end of an SCF/DFT calculation (and at checkpoints for longer runs).

In [1]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("water")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"

# specify name of HDF5 file
scf_drv.filename = "vlx_results_hdf5"

scf_results = scf_drv.compute(molecule, basis)

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.filename = scf_drv.filename

opt_results = opt_drv.compute(molecule, basis, scf_results)

Reading water from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Kohn-Sham                                            
                   Initial Guess Model             : Restart from Checkpoint                                              
                   Convergence Accelerator         : Direct Inversion of Itera

### File content

Depending on the details of calculations, restart files contain different information. A quick overview of the information is readily available.

In [2]:
import h5py

with h5py.File("./vlx_results_hdf5.h5", "r") as f:
    for key in f.keys():
        print(key)

atom_basis_labels_flattened
atom_coordinates
basis_set
dft_func_label
molecular_charge
nuclear_charges
nuclear_repulsion
number_of_alpha_electrons
number_of_atoms
number_of_beta_electrons
opt
potfile_text
scf
spin_multiplicity


## Restarting calculations

VeloxChem can read a restart file in the beginning of the calculation.

In [3]:
import veloxchem as vlx

rsp_drv = vlx.LinearResponseEigenSolver()
rsp_drv.nstates = 5

rsp_drv.filename = "vlx_results_hdf5"

In [4]:
import veloxchem as vlx

scf_drv = vlx.ScfRestrictedDriver()

# specify name of HDF5 file
scf_drv.filename = "vlx_results_hdf5"

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.filename = scf_drv.filename